# 15.3 如何決定找誰處理？


準備好[三組獨立expert](https://birdhackor.github.io/tiny-perceptron-vlm/15.2.html)之後，還缺一件事：同一個token應使用哪一組？可以讓一個小型線性層讀取token的特徵，替每位expert產生分數，再轉成比例。這個分配模組叫router（路由器）。它不是預先寫好的「看到數字就送數學專家」規則，而是可學權重所形成的決定；訓練時需要學習訊號告訴它怎樣分配較有用。

先不做只能選一位的決定，而讓每位expert都輸出，最後按比例混合。假設router替三位產生分數 `[0,0,log(2)]`。softmax把分數取指數，得到 `[1,1,2]`，再除以總和4，比例就是 `[0.25,0.25,0.5]`。若三位對相同輸入 `[1,2]` 分別輸出原值、兩倍與三倍，最後會是原值的 `0.25×1+0.25×2+0.5×3=2.25` 倍，也就是 `[2.25,4.5]`。

這叫soft routing，因為用連續權重混合，而非只留下某個離散選項。它適合先理解router的角色，但每位expert都完成計算，還沒有稀疏計算的節省。下面把router本身設成固定矩陣；三位的輸出也手設成倍數，避免隨機權重干擾核對。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import math
import torch

x = torch.tensor([[1.0, 2.0]])
router_weight = torch.tensor([[0.0, 0.0, math.log(2)], [0.0, 0.0, 0.0]])
scores = x @ router_weight
prob = scores.softmax(-1)
expert_outputs = torch.stack([x, 2 * x, 3 * x], dim=1)
output = (prob[:, :, None] * expert_outputs).sum(1)
print("分配比例", prob.tolist())
print("混合輸出", output.tolist())

`router_weight`有兩列三欄，把兩項特徵轉成三個分數。只有第三欄第一列不為零，因此目前分數由x第一項決定。`torch.stack`在expert方向建立新軸，輸出形狀為一個token、三位expert、兩項特徵。`prob[:,:,None]`在比例後加一個長度一的軸，讓同一expert的比例乘到它的兩項特徵；最後 `sum(1)`沿expert軸合併。輸出應接近上述 `[0.25,0.25,0.5]` 與 `[[2.25,4.5]]`，浮點數末位可能略有差異。

三項比例合計為一，但不能據此說第三位有50%的答對率。它們是混合係數，描述當前路由的分配，並沒有經過答案正確率的校準。router可能偏好某位，是因為該位輸出較合用、數值尺度不同，或目前訓練走進偏斜狀態，需要另外觀察。

練習只把 `router_weight` 第三欄的 `math.log(2)` 改成0。先預測三位各占三分之一，混合倍數變成 `(1+2+3)/3=2`，所以輸出為 `[2,4]`；再執行核對。若你想省運算，接下來還需真的停止執行未選中的expert，不能只把它的混合係數設得很小。
